In [1]:
import pandas as pd
import numpy as np
import nltk
import string
import spacy
from tqdm import tqdm
import os

nltk.download("wordnet")
nltk.download("omw-1.4")  

from nltk.corpus import wordnet as wn


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\malos\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\malos\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
pd.set_option("display.max_colwidth", None)


In [3]:
syns = wn.synsets("program")
print(syns[0].name())
print(syns[0].lemmas()[0].name())
print(syns[0].definition())
print(syns[0].examples())


plan.n.01
plan
a series of steps to be carried out or goals to be accomplished
['they drew up a six-step plan', 'they discussed plans for a new bond issue']


In [4]:
nlp = spacy.load("es_core_news_sm")
punct = set(string.punctuation)

pos_map = {
	"NOUN": "n",
	"VERB": "v",
	"ADJ": "a",
	"ADV": "r"
}

print(punct)


{'@', ']', '`', '}', '%', '<', ')', '^', ',', '/', '>', '(', '+', '\\', '"', '[', '|', "'", ':', '&', '*', '-', '#', '{', '$', ';', '=', '_', '.', '~', '!', '?'}


In [5]:
ALLOWED_CHARS = set(" abcdefghijklmnopqrstuvwxyzñáéíóú")

def normalize(text: str):
	return "".join(
		char for char in text.replace("_", " ").replace("-", " ").lower()
		if char in ALLOWED_CHARS
	)

def extract_synonyms(word: str, wn_pos=None) -> set:
	synonyms = set()
	synsets = wn.synsets(word, lang="spa", pos=wn_pos) if wn_pos else wn.synsets(word, lang="spa")
	
	for syn in synsets:
		for lemma in syn.lemmas(lang="spa"):
			synonyms.add(normalize(lemma.name()))
	
	return synonyms

def get_synonyms(word: str, pos: str):
	word = word.strip()
	word_lower = word.lower()
	
	wn_pos = pos_map.get(pos)
	
	synonyms = extract_synonyms(word_lower, wn_pos)
	synonyms.add(word_lower)
	
	# Si no se encuentran sinónimos, a excepción del original, se intenta eliminar la puntuación
	if len(synonyms) <= 1:
		word_clean = "".join(char for char in word if char not in punct).lower()
		if word_clean != word_lower:
			synonyms.update(extract_synonyms(word_clean, wn_pos))
			synonyms.add(word_clean)

	# Construir distribución de probabilidad: la palabra original obtiene 0.5 y las demás comparten el resto
	n = len(synonyms)
	if n <= 1:
		probabilities = [1.0]
	else:
		probabilities = [0.5 if w == word_lower else 0.5 / (n - 1) for w in synonyms]

	return list(synonyms), probabilities

In [6]:
get_synonyms("hombre", "NOUN")


(['esposo', 'humanidad', 'marido', 'mundo', 'varón', 'hombre'],
 [0.1, 0.1, 0.1, 0.1, 0.1, 0.5])

In [7]:
def match_case(original: str, new: str):
	if original.isupper():
		return new.upper()
	if original[0].isupper():
		return new.capitalize()
	return new


In [8]:
def augment_sentence(sentence: str):
	doc = nlp(sentence)
	orig_text = sentence

	tokens_info = []
	for i, token in enumerate(doc):
		end = token.idx + len(token.text)
		if i < len(doc) - 1:
			next_start = doc[i+1].idx
			# print(end, next_start)
			whitespace = orig_text[end:next_start]
		else:
			whitespace = ""
		tokens_info.append({
			"token": token,
			"whitespace": whitespace
		})

	new_tokens = []
	for info in tokens_info:
		token = info["token"]
		token_text = token.text
		if token.is_punct or token.is_space or token.is_digit or token.is_stop:
			new_tokens.append(token_text)
		else:
			pos = token.pos_
			syns, probs = get_synonyms(token_text, pos)
			chosen = np.random.choice(syns, p=probs)
			chosen = match_case(token.text, chosen.strip())
			new_tokens.append(chosen)

	parts = []
	for info, new_tok in zip(tokens_info, new_tokens):
		parts.append(new_tok + info["whitespace"])
	return "".join(parts)


In [9]:
sentence = "Un  \nhombre."
augment_sentence(sentence)


'Un  \nhombre.'

In [10]:
def augment_row(row: pd.Series, n_augmentations: int):
	sentence1 = row["sentence1"]
	sentence2 = row["sentence2"]
	
	sentence1_variants = [sentence1]
	sentence2_variants = [sentence2]

	for _ in range(n_augmentations):
		sentence1_variants.append(augment_sentence(sentence1))
		sentence2_variants.append(augment_sentence(sentence2))

	# all_pairs = list(product(sentence1_variants, sentence2_variants))
		
	seen = set()
	unique_pairs = []
	for sentence1, sentence2 in zip(sentence1_variants, sentence2_variants):
		pair_key = tuple(sorted([sentence1, sentence2]))
		if pair_key not in seen:
			seen.add(pair_key)
			unique_pairs.append({
				"sentence1": sentence1,
				"sentence2": sentence2,
				"score": row["score"],
				"split": row["split"],
				"score_norm": row["score_norm"]
			})

	new_df = pd.DataFrame(unique_pairs)
	n_new_unique = len(new_df) - 1

	return new_df, n_new_unique


In [11]:
data_path = "../data"
processed_dir = os.path.join(data_path, "processed")
augmented_dir = os.path.join(data_path, "augmented")
os.makedirs(augmented_dir, exist_ok=True)


In [12]:
splits = ["train"]

dfs = []
for split in splits:
	path = os.path.join(processed_dir, f"stsb-es-{split}.csv",)
	df = pd.read_csv(path)
	dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

print(len(df))

5741


In [13]:
augmented_dfs = []
failed_rows = []

total_rows = len(df)
n_augmentations = 2
total_new_examples = 0

for idx, row in tqdm(df.iterrows(), total=total_rows, desc="Augmenting data"):
	new_df, n_new_unique = augment_row(row, n_augmentations=n_augmentations)

	augmented_dfs.append(new_df)
	total_new_examples += n_new_unique

	if n_new_unique <= 0:
		failed_rows.append({
			"index": idx,
			"sentence1": row["sentence1"],
			"sentence2": row["sentence2"]
		})

augmented_data = pd.concat(augmented_dfs, ignore_index=True)

before_global = len(augmented_data)
augmented_data = augmented_data.drop_duplicates(subset=["sentence1", "sentence2"])
after_global = len(augmented_data)

efficiency = total_new_examples / (total_rows * n_augmentations)

print(f"\nAugmentation Summary:")
print(f"- Original dataset size: {total_rows}")
print(f"- Total rows after augmentation (before global dedup): {before_global}")
print(f"- Final dataset size (after global dedup): {after_global}")
print(f"- Total new unique examples added: {after_global - total_rows}")
print(f"- Augmentation efficiency: {efficiency:.2%}")

print(f"\nRows without successful augmentations: {len(failed_rows)}")
if failed_rows:
	failed_df = pd.DataFrame(failed_rows)
	display(failed_df)


Augmenting data: 100%|██████████| 5741/5741 [03:34<00:00, 26.71it/s]



Augmentation Summary:
- Original dataset size: 5741
- Total rows after augmentation (before global dedup): 15284
- Final dataset size (after global dedup): 15234
- Total new unique examples added: 9493
- Augmentation efficiency: 83.11%

Rows without successful augmentations: 422


,index,sentence1,sentence2
0,3,Tres hombres están jugando al ajedrez.,Dos hombres están jugando al ajedrez.
1,5,Algunos hombres están luchando.,Dos hombres están luchando.
2,31,El hombre está tocando la guitarra.,Un hombre está tocando una guitarra.
3,65,Una ardilla está girando en círculos.,Una ardilla corre en círculos.
4,78,Una ardilla corre en círculos.,Una ardilla se mueve en círculos.
...,...,...,...
417,5685,Texas demanda a los refugiados sirios,"Turquía ""explota"" a los refugiados sirios"
418,5687,5 muertos en ataques aéreos israelíes en Gaza,"38 militantes del IS muertos en enfrentamientos, ataques aéreos en Irak"
419,5709,Los artilleros disparan pero no avanzan en la Liga de Campeones,Los clubes de la Liga de Campeones se esfuerzan por evitar los problemas de la Liga de Campeones.
420,5717,Corea del Sur reporta siete nuevos casos de MERS,Mujer surcoreana en contacto con paciente de MERS muere


In [14]:
display(augmented_data.head(20))


,sentence1,sentence2,score,split,score_norm
0,Un avión está despegando.,Un avión está despegando.,5.00,train,1.00
1,Un aeroplano está despegando.,Un avión está despegando.,5.00,train,1.00
2,Un hombre está tocando una gran flauta.,Un hombre está tocando una flauta.,3.80,train,0.76
3,Un varón está tocando una gran flauta.,Un marido está tocando una flauta.,3.80,train,0.76
4,Un hombre está untando queso rallado en una pizza.,Un hombre está untando queso rallado en una pizza cruda.,3.80,train,0.76
5,Un varón está untando queso rallado en una pizza.,Un esposo está untando queso rallado en una pizza cruda.,3.80,train,0.76
6,Tres hombres están jugando al ajedrez.,Dos hombres están jugando al ajedrez.,2.60,train,0.52
7,Un hombre está tocando el violonchelo.,Un hombre sentado está tocando el violonchelo.,4.25,train,0.85
8,Un mundo está tocando el violonchelo.,Un esposo sentado está tocando el violonchelo.,4.25,train,0.85
9,Algunos hombres están luchando.,Dos hombres están luchando.,4.25,train,0.85


In [15]:
display(augmented_data.tail(20))


,sentence1,sentence2,score,split,score_norm
15264,"Francia cierra la mezquita, arresta a un hombre en la represión después de los ataques",La seguridad se reforzó en las iglesias de Nueva Delhi después de los ataques,2.0,train,0.4
15265,"Francia cierra la mezquita, arresta a un esposo en la represión después de los ataques",La determinación se reforzó en las iglesias de Nueva Delhi después de los ataques,2.0,train,0.4
15266,"Francia cierra la mezquita, arresta a un esposo en la represión después de los ataques",La seguridad se reforzó en las iglesias de Nueva Delhi después de los ataques,2.0,train,0.4
15267,Se informa que un avión ruso se ha estrellado en Egipto,Piloto muerto al estrellarse un avión de EE.UU. en Inglaterra,2.0,train,0.4
15268,Se informa que un aeroplano ruso se ha estrellado en Egipto,Piloto muerto al estrellarse un aeroplano de GOBIERNO DE LOS ESTADOS UNIDOS en Inglaterra,2.0,train,0.4
15269,Se informa que un avión ruso se ha estrellado en Egipto,Piloto muerto al estrellarse un aeroplano de EE.UU. en Inglaterra,2.0,train,0.4
15270,Vendavales severos mientras la tormenta Clodagh golpea a Gran Bretaña,Merkel promete la solidaridad de la OTAN con Letonia,0.0,train,0.0
15271,Vendavales severos mientras la tormenta violenta Clodagh golpea a Gran Bretaña,Merkel promete la solidaridad de la ORGANIZACIÓN DEL TRATADO DEL ATLÁNTICO NORTE con Letonia,0.0,train,0.0
15272,Docenas de egipcios rehenes tomados por terroristas libios como venganza por los ataques aéreos,El número de muertos en el accidente de un barco egipcio aumenta a medida que se encuentran más cuerpos en el Nilo,0.0,train,0.0
15273,Docenas de egipcios rehenes tomados por terroristas libios como venganza por los ataques aéreos,El cifra de muertos en el flexión de un barco egipcio aumenta a metro que se encuentran más cuerpos en el Nilo,0.0,train,0.0


In [16]:
for split, data in augmented_data.groupby("split"):
	path = os.path.join(augmented_dir, f"stsb-es-{split}.csv")
	data.to_csv(
		path,
		index=False
	)
	